# Problem 3: Appliance Disaggregation Model (Washing Machine)

**Objective:**
In this notebook, we build, evaluate, and interpret a Non-Intrusive Load Monitoring (NILM) model that estimates **washing machine power consumption** directly from the whole-home **aggregate smart-meter signal** at 1-minute resolution for House 1.

### Key Steps Covered:
1. **Feature Engineering:** Extract rate-of-change, short/medium rolling statistics (5-min, 15-min means and max), and diurnal indicators.
2. **Leakage-Free Validation Design:** Chronological 80/20 train/test split (testing on ~4 months of future unseen data).
3. **Baselines:**
   - *Always-Off Baseline* (predicts 0 W).
   - *Hour-of-Day Mean Usage Profile Baseline*.
4. **Supervised Disaggregation Model:** Gradient Boosted Decision Tree (`HistGradientBoostingRegressor`).
5. **Evaluation & The NILM Paradox:** Analysis of MAE, RMSE, F1-Score, and Total Energy Accuracy (kWh).
6. **Visual Verification:** Actual vs. Predicted power trace and failure mode analysis.

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    precision_score,
    recall_score,
    f1_score
)

os.makedirs('figures', exist_ok=True)
print('Libraries imported successfully!')

## 1. Load Cleaned 1-Minute House 1 Data
We load the 1-minute cleaned data for House 1 and exclude extended sensor outage periods (>5 minutes).

In [ ]:
DATA_PATH = '../house1_cleaned_1min.parquet'
df = pd.read_parquet(DATA_PATH)
df_valid = df[df['outage_flag'] == 0].copy()

print(f'Total valid observations: {len(df_valid):,} minutes')
print(f'Timeline: {df_valid.index.min()} to {df_valid.index.max()}')

## 2. Feature Extraction from Aggregate Signal
We engineer 7 intuitive, physically motivated features:
- `agg`: Current total household power (Watts).
- `agg_diff`: First-order power difference (captures sudden heating element switch-on edges).
- `agg_roll_mean_5`: 5-minute rolling average (smoothes sensor noise).
- `agg_roll_mean_15`: 15-minute rolling average (captures sustained cycle operation).
- `agg_roll_max_15`: 15-minute maximum peak load.
- `hour`: Diurnal time feature (0 to 23).
- `dayofweek`: Weekly pattern feature (0 to 6).

In [ ]:
s_agg = df_valid['Aggregate']
s_wm = df_valid['Appliance5']

features_df = pd.DataFrame(index=df_valid.index)
features_df['agg'] = s_agg
features_df['agg_diff'] = s_agg.diff().fillna(0.0)
features_df['agg_roll_mean_5'] = s_agg.rolling(5, min_periods=1).mean()
features_df['agg_roll_mean_15'] = s_agg.rolling(15, min_periods=1).mean()
features_df['agg_roll_max_15'] = s_agg.rolling(15, min_periods=1).max()
features_df['hour'] = df_valid.index.hour
features_df['dayofweek'] = df_valid.index.dayofweek
features_df['target'] = s_wm
features_df.dropna(inplace=True)

feature_cols = ['agg', 'agg_diff', 'agg_roll_mean_5', 'agg_roll_mean_15', 'agg_roll_max_15', 'hour', 'dayofweek']
print(f'Feature matrix shape: {features_df.shape}')

## 3. Chronological Train / Test Split (Preventing Data Leakage)
Time-series data must never be split with random shuffling because points within the same washing cycle would leak into both training and testing.
We use a **chronological 80/20 split**:

In [ ]:
split_idx = int(len(features_df) * 0.80)
train_df = features_df.iloc[:split_idx]
test_df = features_df.iloc[split_idx:]

X_train, y_train = train_df[feature_cols], train_df['target'].values
X_test, y_test = test_df[feature_cols], test_df['target'].values

print(f'Train: {len(train_df):,} minutes ({train_df.index.min().strftime("%Y-%m-%d")} to {train_df.index.max().strftime("%Y-%m-%d")})')
print(f'Test:  {len(test_df):,} minutes ({test_df.index.min().strftime("%Y-%m-%d")} to {test_df.index.max().strftime("%Y-%m-%d")})')

## 4. Model Training: Baselines vs. Gradient Boosted Trees

In [ ]:
# Baseline 1: Always Off (Zero)
pred_zero = np.zeros(len(y_test))

# Baseline 2: Hour-of-Day Mean Usage Profile
hourly_means = train_df.groupby('hour')['target'].mean().to_dict()
pred_profile = test_df['hour'].map(hourly_means).values

# Disaggregation Model (GBDT)
model = HistGradientBoostingRegressor(max_iter=100, max_depth=8, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)
pred_model = np.clip(model.predict(X_test), 0.0, None)
print('Model training complete!')

## 5. Evaluation Metrics & Comparison
We compute:
- **MAE and RMSE (Watts)**
- **Active State Classification (Precision, Recall, F1-Score at 20W threshold)**
- **Total Energy Estimation Accuracy (%)**

In [ ]:
def evaluate_predictions(y_true, y_pred, name='Model'):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    true_active = (y_true >= 20.0).astype(int)
    pred_active = (y_pred >= 20.0).astype(int)
    prec = precision_score(true_active, pred_active, zero_division=0)
    rec = recall_score(true_active, pred_active, zero_division=0)
    f1 = f1_score(true_active, pred_active, zero_division=0)
    actual_kwh = (np.sum(y_true) / 60.0) / 1000.0
    pred_kwh = (np.sum(y_pred) / 60.0) / 1000.0
    energy_acc = max(0.0, 1.0 - abs(pred_kwh - actual_kwh) / (actual_kwh + 1e-6)) * 100.0
    return {
        'Model': name,
        'MAE (W)': round(mae, 2),
        'RMSE (W)': round(rmse, 2),
        'Precision': round(prec, 3),
        'Recall': round(rec, 3),
        'F1-Score': round(f1, 3),
        'Actual kWh': round(actual_kwh, 2),
        'Predicted kWh': round(pred_kwh, 2),
        'Energy Acc (%)': round(energy_acc, 1)
    }

metrics_table = pd.DataFrame([
    evaluate_predictions(y_test, pred_zero, '1. Always-Off Baseline'),
    evaluate_predictions(y_test, pred_profile, '2. Usage-Profile Baseline'),
    evaluate_predictions(y_test, pred_model, '3. Disaggregation Model (GBDT)')
])
display(metrics_table)

## 6. Visualizations
1. **Actual vs. Predicted Power Trace on Evaluation Data**
2. **Failure Analysis (Overlapping signatures and drain phase errors)**

In [ ]:
test_results = pd.DataFrame(index=test_df.index)
test_results['Aggregate'] = test_df['agg']
test_results['Actual_WM'] = y_test
test_results['Predicted_WM'] = pred_model

# Pick a 2-day evaluation window with active washing
active_test = test_results[test_results['Actual_WM'] > 500]
sample_date = active_test.index[20].strftime('%Y-%m-%d')
window_start = f'{sample_date} 00:00:00'
window_end = pd.to_datetime(sample_date) + pd.Timedelta(days=2)
eval_slice = test_results.loc[window_start:window_end]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax1.plot(eval_slice.index, eval_slice['Aggregate'], label='Total Home Aggregate Power', color='#7f7f7f', alpha=0.5, lw=1)
ax1.plot(eval_slice.index, eval_slice['Actual_WM'], label='Actual Washing Machine (Ground Truth)', color='#1f77b4', lw=2)
ax1.plot(eval_slice.index, eval_slice['Predicted_WM'], label='Predicted Washing Machine (Model)', color='#d62728', lw=1.6, linestyle='--')
ax1.set_title(f'Appliance Disaggregation: Actual vs. Predicted ({sample_date} Window)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Power (Watts)', fontsize=11)
ax1.legend(loc='upper right')
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.fill_between(eval_slice.index, 0, eval_slice['Actual_WM'], color='#1f77b4', alpha=0.25, label='Actual Energy')
ax2.plot(eval_slice.index, eval_slice['Actual_WM'], color='#1f77b4', lw=1.5)
ax2.plot(eval_slice.index, eval_slice['Predicted_WM'], color='#d62728', lw=1.5, label='Model Prediction')
ax2.set_title('Direct Overlay & Cycle Capture', fontsize=11, fontweight='bold')
ax2.set_ylabel('Power (Watts)', fontsize=11)
ax2.set_xlabel('Timestamp', fontsize=11)
ax2.legend(loc='upper right')
ax2.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('figures/problem3_actual_vs_predicted.png', dpi=200)
plt.show()

In [ ]:
# Failure Case Analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

cycle_peak_time = eval_slice[eval_slice['Actual_WM'] > 1000].index[0]
zoom_a = eval_slice.loc[cycle_peak_time - pd.Timedelta(minutes=30):cycle_peak_time + pd.Timedelta(minutes=60)]
ax1.plot(zoom_a.index, zoom_a['Actual_WM'], label='Actual WM', color='#1f77b4', lw=2)
ax1.plot(zoom_a.index, zoom_a['Predicted_WM'], label='Predicted WM', color='#d62728', lw=2, linestyle='--')
ax1.plot(zoom_a.index, zoom_a['Aggregate'], label='Aggregate Power', color='gray', alpha=0.4, lw=1)
ax1.set_title('Success Case: High Peak Capture During Water Heating', fontsize=11, fontweight='bold')
ax1.set_ylabel('Power (Watts)')
ax1.legend(loc='upper right')
ax1.grid(True, linestyle='--', alpha=0.5)

idle_spikes = test_results[(test_results['Aggregate'] > 1800) & (test_results['Actual_WM'] == 0)]
if len(idle_spikes) > 0:
    spike_time = idle_spikes.index[5]
    zoom_b = test_results.loc[spike_time - pd.Timedelta(minutes=30):spike_time + pd.Timedelta(minutes=45)]
    ax2.plot(zoom_b.index, zoom_b['Aggregate'], label='Aggregate (Kettle/Heater)', color='gray', lw=1.2)
    ax2.plot(zoom_b.index, zoom_b['Actual_WM'], label='Actual WM (0 W)', color='#1f77b4', lw=2)
    ax2.plot(zoom_b.index, zoom_b['Predicted_WM'], label='Predicted WM (Ghost Surge)', color='#d62728', lw=2, linestyle='--')
    ax2.set_title('Failure Case: Overlapping Heating Load False Positive', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Power (Watts)')
    ax2.legend(loc='upper right')
    ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('figures/problem3_failure_analysis.png', dpi=200)
plt.show()